# Customer Orders and Retention Analysis
**Beginner portfolio project | SQL + Python + pandas + Matplotlib**

> **Dataset note:** These CSV files contain synthetic practice data generated for this project. They are not real customer records and should not be presented as real company results.

## Business questions
1. How much order value comes from delivered orders?
2. Which products and categories generate the most delivered sales?
3. What is the monthly delivered-sales trend?
4. How many customers place more than one delivered order?
5. Which cities and sales channels have the most orders?

## Tools
Python, pandas, SQLite SQL, Matplotlib, Jupyter Notebook.


## 1. Import libraries and load data

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

# This works when you run the notebook from the project folder.
DATA_DIR = Path("data")
customers = pd.read_csv(DATA_DIR / "customers.csv", parse_dates=["signup_date"])
orders = pd.read_csv(DATA_DIR / "orders.csv", parse_dates=["order_date"])
products = pd.read_csv(DATA_DIR / "products.csv")

print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Products:", products.shape)

display(customers.head())
display(orders.head())
display(products.head())


## 2. Check and clean the data

In [ ]:
# Inspect missing values and duplicate order IDs
print("Missing values in orders:")
display(orders.isna().sum().to_frame("missing_values"))
print("Duplicate order IDs:", orders["order_id"].duplicated().sum())

# Standardize text fields
orders["sales_channel"] = orders["sales_channel"].astype("string").str.strip().str.title()
orders["order_status"] = orders["order_status"].astype("string").str.strip().str.title()

# Ensure numeric columns have numeric types
for col in ["quantity", "unit_price", "discount_pct", "order_value"]:
    orders[col] = pd.to_numeric(orders[col], errors="coerce")

# Recalculate order value so the calculation is explicit and reproducible
orders["calculated_order_value"] = (
    orders["quantity"] * orders["unit_price"] * (1 - orders["discount_pct"])
).round(2)

print("Rows after cleaning:", len(orders))
print("Order status values:", sorted(orders["order_status"].dropna().unique()))
print("Sales channel values:", sorted(orders["sales_channel"].dropna().unique()))


## 3. Join orders to customers and products

In [ ]:
analysis = (
    orders.merge(customers, on="customer_id", how="left")
          .merge(products[["product_id", "product_name", "category"]], on="product_id", how="left")
)

print("Analysis rows:", len(analysis))
display(analysis.head())


## 4. SQL analysis with SQLite

In [ ]:
# Load the dataframes into an in-memory SQLite database
conn = sqlite3.connect(":memory:")
customers.to_sql("customers", conn, index=False, if_exists="replace")
orders.to_sql("orders", conn, index=False, if_exists="replace")
products.to_sql("products", conn, index=False, if_exists="replace")

# Query 1: order counts by status
status_sql = '''
SELECT order_status,
       COUNT(*) AS order_count,
       ROUND(SUM(order_value), 2) AS total_order_value
FROM orders
GROUP BY order_status
ORDER BY order_count DESC;
'''
display(pd.read_sql_query(status_sql, conn))

# Query 2: delivered sales by product category
category_sql = '''
SELECT p.category,
       COUNT(o.order_id) AS delivered_orders,
       ROUND(SUM(o.order_value), 2) AS delivered_sales
FROM orders AS o
JOIN products AS p ON o.product_id = p.product_id
WHERE o.order_status = 'Delivered'
GROUP BY p.category
ORDER BY delivered_sales DESC;
'''
display(pd.read_sql_query(category_sql, conn))

# Query 3: top 10 customers by delivered order value
top_customers_sql = '''
SELECT c.customer_id, c.customer_name,
       COUNT(o.order_id) AS delivered_orders,
       ROUND(SUM(o.order_value), 2) AS delivered_sales
FROM customers AS c
JOIN orders AS o ON c.customer_id = o.customer_id
WHERE o.order_status = 'Delivered'
GROUP BY c.customer_id, c.customer_name
ORDER BY delivered_sales DESC
LIMIT 10;
'''
display(pd.read_sql_query(top_customers_sql, conn))


## 5. Monthly delivered-sales trend

In [ ]:
delivered = analysis[analysis["order_status"] == "Delivered"].copy()
delivered["order_month"] = delivered["order_date"].dt.to_period("M").astype(str)

monthly_sales = (
    delivered.groupby("order_month", as_index=False)["order_value"]
    .sum()
    .rename(columns={"order_value": "delivered_sales"})
)

display(monthly_sales)

plt.figure(figsize=(11, 5))
plt.plot(monthly_sales["order_month"], monthly_sales["delivered_sales"], marker="o")
plt.title("Monthly Delivered Sales")
plt.xlabel("Month")
plt.ylabel("Delivered sales (sample currency units)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 6. Repeat-customer analysis

In [ ]:
delivered_orders_per_customer = (
    delivered.groupby("customer_id")["order_id"]
    .nunique()
    .reset_index(name="delivered_order_count")
)

repeat_summary = pd.DataFrame({
    "metric": ["Customers with at least one delivered order",
               "Customers with 2+ delivered orders",
               "Repeat-customer share among purchasing customers"],
    "value": [
        delivered_orders_per_customer["customer_id"].nunique(),
        int((delivered_orders_per_customer["delivered_order_count"] >= 2).sum()),
        round((delivered_orders_per_customer["delivered_order_count"] >= 2).mean() * 100, 1)
    ]
})
display(repeat_summary)

repeat_counts = delivered_orders_per_customer["delivered_order_count"].value_counts().sort_index()
plt.figure(figsize=(8, 4))
plt.bar(repeat_counts.index.astype(str), repeat_counts.values)
plt.title("Number of Delivered Orders per Customer")
plt.xlabel("Delivered orders per customer")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()


## 7. Channel and city analysis

In [ ]:
channel_summary = (
    analysis.groupby("sales_channel", as_index=False)
    .agg(total_orders=("order_id", "nunique"),
         delivered_orders=("order_status", lambda s: (s == "Delivered").sum()),
         all_order_value=("order_value", "sum"))
    .sort_values("total_orders", ascending=False)
)
display(channel_summary)

city_summary = (
    delivered.groupby("city", as_index=False)
    .agg(delivered_orders=("order_id", "nunique"),
         delivered_sales=("order_value", "sum"))
    .sort_values("delivered_sales", ascending=False)
)
display(city_summary)

plt.figure(figsize=(8, 4))
plt.bar(city_summary["city"], city_summary["delivered_sales"])
plt.title("Delivered Sales by City")
plt.xlabel("City")
plt.ylabel("Delivered sales (sample currency units)")
plt.tight_layout()
plt.show()


## 8. Write evidence-based findings

In [ ]:
# These are calculated from the sample data; don't copy guessed findings into the CV.
total_orders = analysis["order_id"].nunique()
delivered_count = int((analysis["order_status"] == "Delivered").sum())
returned_count = int((analysis["order_status"] == "Returned").sum())
cancelled_count = int((analysis["order_status"] == "Cancelled").sum())
delivered_sales = round(delivered["order_value"].sum(), 2)
repeat_pct = round((delivered_orders_per_customer["delivered_order_count"] >= 2).mean() * 100, 1)

print(f"Total orders: {total_orders}")
print(f"Delivered orders: {delivered_count}")
print(f"Returned orders: {returned_count}")
print(f"Cancelled orders: {cancelled_count}")
print(f"Delivered sales: {delivered_sales:,.2f} sample currency units")
print(f"Repeat-customer share among customers with delivered orders: {repeat_pct}%")
print("\nTop category by delivered sales:")
display(pd.read_sql_query(category_sql, conn).head(1))
print("\nBest month by delivered sales:")
display(monthly_sales.sort_values("delivered_sales", ascending=False).head(1))


## 9. Project conclusions

These conclusions are based on the **synthetic practice dataset** included with this project. They are not real business results.

1. The dataset contains **1,000 orders**: **789 delivered (78.9%)**, **125 returned (12.5%)**, and **86 cancelled (8.6%)**.
2. **Furniture** generated the highest delivered sales, at **1,252,730 sample currency units**.
3. **February 2025** was the strongest month for delivered sales, at **330,983.50 sample currency units**.
4. **Dhaka** had the highest delivered sales among the listed cities, and the **Website** channel had the most orders.
5. **95.5%** of customers with at least one delivered order had two or more delivered orders in this sample. This high share reflects how the synthetic dataset was generated and should not be treated as a real-world retention benchmark.

### Recommendations for a real business
- Investigate reasons for returns and cancellations, including product, delivery, and payment issues.
- Review Furniture's sales performance and check profitability and stock availability before making inventory decisions.
- Compare monthly sales with promotions, seasonality, and product availability.
- Calculate retention and repeat-purchase rates from verified real customer data before making business claims.


In [ ]:
conn.close()
print("Analysis complete. Save the notebook after all cells run successfully.")
